<a href="https://colab.research.google.com/github/PercyTechX/del-excel-al-ml/blob/main/temporada-2/08-etica-auditar-una-decision/clase.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Episodio 2.8 · Ética I: auditar una decisión

**Serie:** Del Excel al Machine Learning · **Temporada 2** · 🎬 Video próximamente

## 🎯 Qué vas a aprender

- **Auditar** una decisión: comparar tasas por grupo y por cargo.
- Qué es un **indicador indirecto**.
- Por qué **no hay que concluir con pocos casos**.

## 📊 En Excel lo harías así

Con una tabla dinámica: en filas el cargo, en columnas el sexo, y como valor el promedio de una columna con 1 si llamaron a la persona a entrevista.

Para esto no hace falta Machine Learning. Y es lo primero que conviene hacer antes de automatizar cualquier decisión sobre personas.

## 🐍 En Python

### Paso 1: los datos, y una advertencia

`postulaciones.csv` tiene postulaciones de 2022 a 2025, y si el reclutador llamó a cada persona a entrevista.

> ⚠️ **Este archivo es inventado, y el sesgo contra las mujeres lo programamos a propósito para aprender a detectarlo.** No describe a ninguna empresa ni a ninguna persona real. Tampoco dice nada sobre la capacidad de nadie. `sexo` solo usa `F` y `M`, una simplificación de estos datos.

Además, son muchas postulaciones para una empresa de este tamaño: es un ejemplo para aprender.

In [1]:
import pandas as pd

DATOS = "https://raw.githubusercontent.com/PercyTechX/del-excel-al-ml/main/datos/"

In [2]:
postulaciones = pd.read_csv(DATOS + "postulaciones.csv")

print(len(postulaciones))

postulaciones.head()

3200


,id_postulacion,fecha,cargo,sede,sexo,anios_experiencia,meses_sin_empleo,puede_viajar,puntaje_prueba,entrevistado
0,A00001,2022-01-01,Asistente administrativo,Trujillo,M,1.4,2,1,50,0
1,A00002,2022-01-01,Almacenero,Arequipa,M,5.8,1,0,58,0
2,A00003,2022-01-01,Asistente administrativo,Lima,F,5.8,13,0,67,0
3,A00004,2022-01-02,Repartidor,Trujillo,M,1.3,0,1,70,1
4,A00005,2022-01-03,Almacenero,Chiclayo,M,1.3,1,0,52,0


### Paso 2: ¿quién mide lo mismo?

Antes de comparar, buscamos algo justo. Aquí, el puntaje de la prueba de la empresa. Lo programamos igual para todos. **Eso no se puede suponer de una prueba real**, que también puede estar sesgada: hay que comprobarlo.

In [3]:
postulaciones.groupby("sexo")[["puntaje_prueba", "entrevistado"]].mean().round(2)

,puntaje_prueba,entrevistado
sexo,,
F,61.50,0.14
M,62.31,0.32


Casi el mismo puntaje promedio, y una tasa de entrevista muy distinta.

Pero comparar el total puede engañar, porque hombres y mujeres postulan a cargos distintos. Comparamos **dentro de cada cargo, y entre quienes sacaron 75 o más**.

In [4]:
def llamados(tabla):
    """Por cargo y sexo: a cuántas personas se llamó, de cuántas, y qué parte."""
    return tabla.pivot_table(
        index="cargo",
        columns="sexo",
        values="entrevistado",
        aggfunc=["sum", "size", "mean"],
    ).round(2)

In [5]:
altos = postulaciones[postulaciones["puntaje_prueba"] >= 75]

llamados(altos)

sum     size       mean      
sexo                       F   M    F    M     F     M
cargo                                                 
Almacenero                10  77   46  137  0.22  0.56
Asistente administrativo  22  22   57   34  0.39  0.65
Repartidor                 3  70   18  132  0.17  0.53
Vendedor                  17  51   87   95  0.20  0.54

`sum` es a cuántas personas se llamó, `size` de cuántas, y `mean` qué parte. En los cuatro cargos, las mujeres quedan por debajo.

### Paso 3: el caso de Asistente

En este archivo solo programamos sesgo directo en Vendedor, Almacenero y Repartidor. **En Asistente, no.** Y hay brecha igual. ¿De dónde sale?

In [6]:
asistentes = postulaciones[postulaciones["cargo"] == "Asistente administrativo"]

asistentes.groupby("sexo")[["meses_sin_empleo", "puede_viajar"]].mean().round(2)

,meses_sin_empleo,puede_viajar
sexo,,
F,12.98,0.35
M,4.32,0.70


Esas dos columnas difieren mucho entre hombres y mujeres. Ahora comparamos perfiles parecidos, separando por las dos columnas a la vez:

In [7]:
perfil = asistentes.assign(pausa_larga=asistentes["meses_sin_empleo"] >= 12)

perfil.pivot_table(
    index=["puede_viajar", "pausa_larga"],
    columns="sexo",
    values="entrevistado",
    aggfunc=["size", "mean"],
).round(2)

size       mean      
sexo                        F    M     F     M
puede_viajar pausa_larga                      
0            False        148   58  0.17  0.22
             True         102    3  0.09  0.00
1            False         82  133  0.44  0.38
             True          52    8  0.13  0.38

Dentro de un mismo perfil, la brecha casi desaparece, salvo en los grupos con muy pocos casos, que no sirven para concluir. Y **lo sabemos porque lo programamos**: el reclutador castiga los meses sin empleo y premia poder viajar. Con datos reales, una tabla así da una pista, no una prueba. Estas columnas se llaman **indicadores indirectos**: sin mencionar el sexo, lo delatan.

Cuidado con cómo se lee esto:

- Son valores programados. Este archivo no puede decir **por qué** ocurren, ni si predicen desempeño.
- No miden capacidad.
- **Hay que preguntar si cada requisito hace falta.** Poder viajar puede ser necesario para un repartidor o un vendedor. No se borra sin analizarlo, y no se da por bueno sin analizarlo.

### Paso 4: pocos casos engañan

Mira Repartidor: son 3 mujeres llamadas de 18. Si una más hubiera sido llamada, la tasa cambiaría varios puntos. Volvamos a Asistente, ahora con todos los puntajes y abriendo por año:

In [8]:
tabla = postulaciones.assign(anio=postulaciones["fecha"].str[:4])

tabla[tabla["cargo"] == "Asistente administrativo"].pivot_table(
    index="anio",
    columns="sexo",
    values="entrevistado",
    aggfunc=["sum", "size", "mean"],
).round(2)

sum     size      mean      
sexo   F   M    F   M     F     M
anio                             
2022  12  18   61  56  0.20  0.32
2023  28  14  126  50  0.22  0.28
2024  15  16  100  46  0.15  0.35
2025  22  19   97  50  0.23  0.38

Un año suelto tiene pocos casos, y el tamaño de la brecha cambia mucho: en 2023 son 22 % contra 28 %, y en 2024 son 15 % contra 35 %. Parte de eso es azar. También lo son los 26 puntos de Asistente de la primera tabla: en otras versiones de estos datos, lo esperado son unos 16. Por eso:

- **Mira siempre los conteos**, no solo el porcentaje.
- **Acumula varios años** antes de concluir.
- **Si la dirección se repite** en todos los cargos, como aquí, ya es una señal seria que merece una revisión.

### ¿Qué hacer con esto?

Una tabla dinámica ya mostró el problema. **No hizo falta un modelo.** Lo que sigue es conversar con quienes deciden, revisar los requisitos y volver a mirar cada cierto tiempo.

En el próximo episodio veremos qué pasa si alguien propone entrenar un modelo con estos datos.

> Las normas sobre empleo y datos personales se deben consultar en la fuente oficial. Este video no es asesoría legal.

## 🤖 Pídeselo a la IA

> *"Quiero revisar si un proceso de selección trata igual a hombres y mujeres, con datos anónimos e inventados para practicar. ¿Qué preguntas debería hacerme antes de sacar conclusiones de una tabla dinámica?"*

La IA te ayuda a escribir y entender código, pero **tú debes entender lo que hace**. Revisa cada respuesta antes de usarla, y no pegues datos de tu empresa.

## ✅ Resumen

| En Excel | En Python |
|---|---|
| Tabla dinámica por cargo y sexo | `tabla.pivot_table(index=..., columns=..., aggfunc=[...])` |
| Filtrar por puntaje | `tabla[tabla["puntaje_prueba"] >= 75]` |
| Ver el conteo, no solo el porcentaje | `aggfunc=["sum", "size", "mean"]` |

## 📝 Ahora te toca

Abre [el ejercicio en Colab](https://colab.research.google.com/github/PercyTechX/del-excel-al-ml/blob/main/temporada-2/08-etica-auditar-una-decision/ejercicio.ipynb) y haz tu propia auditoría. Cuando termines, compara con [la solución](https://colab.research.google.com/github/PercyTechX/del-excel-al-ml/blob/main/temporada-2/08-etica-auditar-una-decision/solucion.ipynb).

---

*Del Excel al Machine Learning* · por [Percy Nazario](https://www.linkedin.com/in/percy-nazario/) · [Repositorio](https://github.com/PercyTechX/del-excel-al-ml) · Contenido bajo licencia CC BY 4.0